[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 01](README.md)

# Modelos de paralelismo y descomposición

**Tema:** 01 · **Sesiones:** 3 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Qué parte del trabajo puede ejecutarse simultáneamente y qué dependencias fijan el camino crítico?


## Resultados de aprendizaje

- Distinguir concurrencia, paralelismo de datos y de tareas.
- Representar dependencias mediante un DAG.
- Calcular trabajo, span y paralelismo promedio.


## Modelo conceptual

El trabajo T1 suma el costo de todas las tareas; el span T∞ es el camino dependiente más largo.

La aceleración con p recursos está acotada por min(p, T1/T∞), aun sin costos de comunicación.

Una descomposición correcta conserva dependencias y evita crear más coordinación que cómputo útil.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "01"
NOTEBOOK = "01_fundamentos/01_modelos.ipynb"
assert (ROOT / "curso" / "notebooks" / "01_fundamentos" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Trabajo y camino crítico

Se evalúa un DAG pequeño con duraciones explícitas.


In [ ]:
tasks = {"leer": 2, "partir": 1, "A": 5, "B": 4, "combinar": 2}
predecessors = {"leer": [], "partir": ["leer"], "A": ["partir"], "B": ["partir"], "combinar": ["A", "B"]}
finish = {}
for task in tasks:
    start = max((finish[p] for p in predecessors[task]), default=0)
    finish[task] = start + tasks[task]
work = sum(tasks.values())
span = max(finish.values())
parallelism = work / span
assert (work, span) == (14, 10)
print({"work": work, "span": span, "parallelism": round(parallelism, 2), "finish": finish})


**Interpretación.** A y B son concurrentes, pero lectura, partición y combinación permanecen en el camino crítico.


## Partición balanceada

Se distribuyen n elementos sin perder ni duplicar índices.


In [ ]:
def ranges(n, workers):
    q, r = divmod(n, workers)
    start = 0
    result = []
    for worker in range(workers):
        size = q + (worker < r)
        result.append((start, start + size))
        start += size
    return result
chunks = ranges(23, 4)
covered = [i for begin, end in chunks for i in range(begin, end)]
assert covered == list(range(23))
print(chunks)


**Interpretación.** El resto se reparte de forma determinista y la cobertura constituye una prueba simple de corrección.


## Práctica reproducible

1. Dibujar el DAG de una operación del curso.
2. Identificar T1, T∞ y la granularidad de cada nodo.
3. Proponer una descomposición y señalar la sincronización necesaria.


## Errores frecuentes

- Confundir más tareas con mayor paralelismo.
- Omitir dependencias de datos.
- Evaluar únicamente el tiempo paralelo sin referencia serial.

## Criterios de aceptación

- DAG acíclico y dependencias justificadas.
- Cobertura de datos sin solapamientos involuntarios.
- Cotas de aceleración calculadas antes de medir.


## Referencias y material relacionado

- [Planeación: fundamentos](../../../docs/PLANEACION_CURSO.md#6-calendario-de-38-sesiones)
- [Índice del curso](../../../INDICE_CURSO.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 01](README.md)
